Build a Simple OpenAI LLM based Calculator using Tool Calling techenique.

We have four Tools defined - Addition, Subtract, Multiply, Divide. Tool interactions are managed by 'ToolNode' package in LangGraph.

In [ ]:
# Import Python Packages
import os
import keyboard
from dotenv import load_dotenv

from typing import TypedDict, Literal
from IPython.display import Image, display

# Import LangGraph Lib
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, AnyMessage

# Import OpenAI Lib
from langchain_openai import ChatOpenAI

load_dotenv()

In [ ]:
# Calculator Operations

# Addition
def addition(num1: int, num2: int) -> int:
    "Add Two Numbers. Args: num1 and num2"
    return(num1 + num2)

# Subtract  
def subtract(num1: int, num2: int) -> int:
    "Subtract Two Numbers. Args: num1 and num2"
    if (num1 >= num2):
        r1 = num1 - num2
    else:
        r1 = num2 - num1
    
    return(r1)

# Multiply
def multiply(num1: int, num2: int) -> int:
    "Multiply Two Numbers. Args: num1 and num2"
    return(num1 * num2)

# Division
def divide(num1: int, num2: int) -> int:
    "Divide Two Numbers. Args: num1 and num2"
    if ((num1 > 0) & (num2 > 0)):
        r1 = num1 > num2
    else:
        r1 = 0

    return(r1)


In [ ]:
# Instantiate ChatOpenAI & Bind Calculator Operations

chat_llm = ChatOpenAI(model='gpt-4o', temperature=0, max_retries=2)
chat_llm_with_tool = chat_llm.bind_tools([addition, subtract, multiply, divide])

In [ ]:
# LangGraph State Object & Functions

# State Object. 
# For ToolNode (below) to work, response of LLM should be set to 'messages' variable.
# This 'messages' variable need to be part of State Object that will be input for ToolNode.
# class State(TypedDict):
#    chat_message: AnyMessage
#    messages: list[AnyMessage]
# So if we are making a Calculator with Chatting interface. 
# It will be better to have only one 'messages' variables of 'list' Type

class State(TypedDict):
    messages: list[AnyMessage]

# LLM Invoke function
def chat_llm_with_toolnode(state: State) -> AnyMessage:
    print(state['messages'])
    ai_message = chat_llm_with_tool.invoke(state['messages'])
    print(ai_message)
    return({'messages': [ai_message]})


In [ ]:
# Build the LangGraph Flow

graph_builder = StateGraph(state_schema=State)

# Add Nodes
graph_builder.add_node("llm_node", chat_llm_with_toolnode)
graph_builder.add_node("tools", ToolNode([addition, subtract, multiply, divide]))  # Name of ToolNode SHOULD be "tools" always

# Add Edges
graph_builder.add_edge(START, "llm_node")
graph_builder.add_conditional_edges("llm_node", tools_condition) # tools_condition always expect the target nodename to be "tools"
graph_builder.add_edge("tools", END)  # If tool calling required, condition Node return Route to 'tools' Node.

graph = graph_builder.compile()

# Visualize Graph
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# Invoke LLM with Chat Calculator Tools

user_prompt = input("User: ")
print(graph.invoke({'messages': [HumanMessage(content=user_prompt)]}))

user_prompt = input("User: ")
print(graph.invoke({'messages': [HumanMessage(content=user_prompt)]}))

In [ ]:
#Scratch Pad
#graph.invoke({'messages': [HumanMessage(content="Add 7 and 10")]})

# Sample Response 

# -- Below Content is populated for LLM Response.
#[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]
#content='Hello! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 149, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-C0Z4lIfmPGyuMkAw1fHJvTt5tiJ0M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--5f028998-f532-44d8-88c5-b8a80c334e45-0' usage_metadata={'input_tokens': 149, 'output_tokens': 10, 'total_tokens': 159, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
#{'messages': [AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 149, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-C0Z4lIfmPGyuMkAw1fHJvTt5tiJ0M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5f028998-f532-44d8-88c5-b8a80c334e45-0', usage_metadata={'input_tokens': 149, 'output_tokens': 10, 'total_tokens': 159, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

# -- Below Content is empty when the call from LLM is a ToolCall.
#[HumanMessage(content='What is teh sum of 10 and 15 ?', additional_kwargs={}, response_metadata={})]
#content='' additional_kwargs={'tool_calls': [{'id': 'call_mHq50LPRcbDZcpEuVP3GBGnm', 'function': {'arguments': '{"num1":10,"num2":15}', 'name': 'addition'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 159, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-C0Z5D4new657krQAFHCHfhHT5Ortt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--04c21bde-ae07-45ab-a4b6-8eb1c62cc8d3-0' tool_calls=[{'name': 'addition', 'args': {'num1': 10, 'num2': 15}, 'id': 'call_mHq50LPRcbDZcpEuVP3GBGnm', 'type': 'tool_call'}] usage_metadata={'input_tokens': 159, 'output_tokens': 20, 'total_tokens': 179, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
#{'messages': [ToolMessage(content='25', name='addition', tool_call_id='call_mHq50LPRcbDZcpEuVP3GBGnm')]}